# Smart Power — SQL Analysis

This notebook demonstrates the database layer of the project. It uses MySQL 8 and SQL to:

1. validate the clean hourly table created by the transformation notebook;
2. calculate dynamic price and carbon-intensity quartile thresholds;
3. create reusable views for Tableau;
4. verify that SQL results match the EDA notebook.

**Definitions**
- **Cheap:** electricity price is in the bottom quartile (≤ the 25th percentile).
- **Low-carbon:** carbon intensity is in the bottom quartile (≤ the 25th percentile).
- **Cheap and low-carbon:** both conditions are true for the same hour. Renewable share remains an explanatory variable.

> MySQL stores the source timestamps in UTC. Tableau should convert `timestamp_utc` to `Europe/Amsterdam` so daylight-saving time is handled correctly.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sqlalchemy import text

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from database import get_mysql_engine

CREATE_VIEWS_SQL = PROJECT_ROOT / "sql" / "01_create_views.sql"
QUALITY_SQL = PROJECT_ROOT / "sql" / "02_quality_checks.sql"

assert CREATE_VIEWS_SQL.exists(), f"SQL file not found: {CREATE_VIEWS_SQL}"
assert QUALITY_SQL.exists(), f"SQL file not found: {QUALITY_SQL}"

engine = get_mysql_engine(PROJECT_ROOT)
connection = engine.connect()
print("Connected to MySQL database: smart_power")

Connected to MySQL database: smart_power


## 1. Create the Tableau-ready views

The SQL is kept in a separate `.sql` file so it can also be opened and run directly in MySQL Workbench. `DROP VIEW IF EXISTS` makes this step repeatable.

In [2]:
create_views_sql = CREATE_VIEWS_SQL.read_text(encoding="utf-8")

# The file contains ordinary MySQL statements separated by semicolons.
sql_without_comments = "\n".join(
    line for line in create_views_sql.splitlines()
    if not line.strip().startswith("--")
)
statements = [statement.strip() for statement in sql_without_comments.split(";") if statement.strip()]

with engine.begin() as script_connection:
    for statement in statements:
        script_connection.execute(text(statement))

print("MySQL views created successfully.")

MySQL views created successfully.


## 2. Inspect the database objects

In [3]:
objects_query = """
SELECT
    table_name AS name,
    table_type AS type
FROM information_schema.tables
WHERE table_schema = DATABASE()
ORDER BY type, name;
"""
pd.read_sql_query(objects_query, connection)

,name,type
0,hourly_data,BASE TABLE
1,vw_analysis_thresholds,VIEW
2,vw_daily_summary_utc,VIEW
3,vw_hourly_dashboard,VIEW
4,vw_hourly_summary_utc,VIEW


## 3. Data-quality checks

A valid hourly table must have one row per timestamp, no missing core measurements, and renewable shares between 0 and 1.

In [4]:
quality_query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT timestamp_utc) AS unique_hours,
    SUM(CASE
        WHEN electricity_price IS NULL
          OR renewable_share IS NULL
          OR renewable_generation IS NULL
          OR total_generation IS NULL
          OR carbon_intensity_gco2_kwh IS NULL
          OR wind_speed IS NULL
          OR solar_radiation IS NULL
        THEN 1 ELSE 0 END) AS incomplete_rows,
    MIN(timestamp_utc) AS first_hour_utc,
    MAX(timestamp_utc) AS last_hour_utc,
    MIN(renewable_share) AS min_renewable_share,
    MAX(renewable_share) AS max_renewable_share
FROM hourly_data;
"""
quality = pd.read_sql_query(quality_query, connection)
display(quality)

assert quality.loc[0, "total_rows"] == quality.loc[0, "unique_hours"]
assert quality.loc[0, "incomplete_rows"] == 0
assert 0 <= quality.loc[0, "min_renewable_share"] <= 1
assert 0 <= quality.loc[0, "max_renewable_share"] <= 1
print("All data-quality checks passed.")

,total_rows,unique_hours,incomplete_rows,first_hour_utc,last_hour_utc,min_renewable_share,max_renewable_share
0,1170,1170,0.0,2026-05-28,2026-08-03 21:00:00,0.000164,0.690521


All data-quality checks passed.


In [5]:
duplicate_query = """
SELECT timestamp_utc, COUNT(*) AS duplicate_count
FROM hourly_data
GROUP BY timestamp_utc
HAVING COUNT(*) > 1;
"""
duplicates = pd.read_sql_query(duplicate_query, connection)
assert duplicates.empty
print("Duplicate timestamps: 0")

Duplicate timestamps: 0


## 4. Dynamic cheap and low-carbon thresholds

The view calculates linearly interpolated quartiles in SQL. These values should match `pandas.quantile()` in the EDA notebook.

In [6]:
thresholds = pd.read_sql_query(
    "SELECT * FROM vw_analysis_thresholds;", connection
)
display(thresholds.style.format({
    "cheap_price_threshold": "€{:.3f}/kWh",
    "low_carbon_threshold_gco2_kwh": "{:.1f}",
}))

assert abs(thresholds.loc[0, "cheap_price_threshold"] - 0.08) < 1e-9
assert thresholds.loc[0, "low_carbon_threshold_gco2_kwh"] >= 0
print("SQL thresholds match the EDA notebook.")

,cheap_price_threshold,low_carbon_threshold_gco2_kwh
0,€0.080/kWh,341.8


SQL thresholds match the EDA notebook.


## 5. Validate the Tableau row-level view

In [7]:
dashboard_check_query = """
SELECT
    COUNT(*) AS dashboard_rows,
    SUM(is_cheap) AS cheap_hours,
    SUM(is_low_carbon) AS low_carbon_hours,
    SUM(is_cheap_and_low_carbon) AS cheap_and_low_carbon_hours
FROM vw_hourly_dashboard;
"""
dashboard_check = pd.read_sql_query(dashboard_check_query, connection)
display(dashboard_check)

assert dashboard_check.loc[0, "dashboard_rows"] == quality.loc[0, "total_rows"]
assert dashboard_check.loc[0, "cheap_and_low_carbon_hours"] > 0
print("Dashboard view contains cheap-and-low-carbon hours.")

,dashboard_rows,cheap_hours,low_carbon_hours,cheap_and_low_carbon_hours
0,1170,302.0,293.0,164.0


Dashboard view contains cheap-and-low-carbon hours.


In [8]:
preview_query = """
SELECT
    timestamp_utc, electricity_price, renewable_share,
    carbon_intensity_gco2_kwh, wind_speed, solar_radiation,
    is_cheap, is_low_carbon, is_cheap_and_low_carbon
FROM vw_hourly_dashboard
WHERE is_cheap_and_low_carbon = 1
ORDER BY electricity_price ASC, renewable_share DESC
LIMIT 10;
"""
pd.read_sql_query(preview_query, connection).style.format({
    "electricity_price": "€{:.3f}",
    "renewable_share": "{:.1%}",
})

,timestamp_utc,electricity_price,renewable_share,carbon_intensity_gco2_kwh,wind_speed,solar_radiation,is_cheap,is_low_carbon,is_cheap_and_low_carbon
0,2026-07-12 12:00:00,€-0.020,56.8%,286.690000,11.500000,818.000000,1,1,1
1,2026-07-12 13:00:00,€-0.010,56.7%,283.322500,12.200000,720.000000,1,1,1
2,2026-06-07 07:00:00,€-0.010,35.5%,236.602500,20.900000,7.000000,1,1,1
3,2026-06-07 08:00:00,€-0.010,31.4%,285.292500,19.800000,38.000000,1,1,1
4,2026-06-07 09:00:00,€-0.010,28.2%,323.615000,18.400000,39.000000,1,1,1
5,2026-07-04 12:00:00,€-0.010,17.0%,110.902500,13.000000,493.000000,1,1,1
6,2026-07-04 11:00:00,€-0.010,14.6%,108.912500,15.800000,246.000000,1,1,1
7,2026-07-19 07:00:00,€0.000,68.1%,113.942500,16.600000,316.000000,1,1,1
8,2026-07-19 09:00:00,€0.000,66.7%,115.110000,16.200000,539.000000,1,1,1
9,2026-07-19 08:00:00,€0.000,66.1%,117.872500,16.900000,453.000000,1,1,1


## 6. Tableau-ready summary views

The row-level dashboard view is the primary Tableau source. The two summary views are useful for quick SQL checks and pre-aggregated charts. Hours shown here are UTC; the EDA recommendation uses Dutch local time.

In [9]:
hourly_summary = pd.read_sql_query(
    "SELECT * FROM vw_hourly_summary_utc ORDER BY hour_utc;", connection
)
display(hourly_summary.style.format({
    "avg_price": "€{:.3f}",
    "avg_renewable_share": "{:.1%}",
    "cheap_low_carbon_rate": "{:.1%}",
}))

,hour_utc,observed_hours,avg_price,avg_renewable_share,avg_carbon_intensity_gco2_kwh,cheap_low_carbon_hours,cheap_low_carbon_rate
0,0,48,€0.156,20.0%,420.144010,1.000000,2.1%
1,1,48,€0.151,20.5%,417.900104,2.000000,4.2%
2,2,48,€0.150,21.4%,414.538542,2.000000,4.2%
3,3,48,€0.156,21.2%,411.972396,2.000000,4.2%
4,4,49,€0.166,19.7%,416.735306,2.000000,4.1%
5,5,50,€0.161,19.1%,414.652500,3.000000,6.0%
6,6,50,€0.145,19.0%,412.975450,6.000000,12.0%
7,7,50,€0.111,20.3%,399.466600,10.000000,20.0%
8,8,49,€0.077,22.8%,391.049745,12.000000,24.5%
9,9,49,€0.054,23.7%,381.813929,14.000000,28.6%


In [10]:
daily_summary = pd.read_sql_query("""
SELECT *
FROM vw_daily_summary_utc
ORDER BY date_utc DESC
LIMIT 10;
""", connection)
daily_summary.style.format({
    "avg_price": "€{:.3f}", "min_price": "€{:.3f}", "max_price": "€{:.3f}",
    "avg_renewable_share": "{:.1%}",
})

,date_utc,observed_hours,avg_price,min_price,max_price,avg_renewable_share,avg_carbon_intensity_gco2_kwh,avg_wind_speed,avg_solar_radiation,cheap_low_carbon_hours
0,2026-08-03,22,€0.148,€0.000,€0.240,22.1%,429.234091,12.495455,312.681818,2.000000
1,2026-08-02,24,€0.125,€0.000,€0.230,19.1%,466.394271,12.325000,282.375000,0.000000
2,2026-08-01,24,€0.142,€0.000,€0.220,4.4%,565.256354,7.308333,287.583333,0.000000
3,2026-07-31,24,€0.176,€0.100,€0.230,12.9%,522.345521,9.429167,203.375000,0.000000
4,2026-07-30,24,€0.153,€0.020,€0.240,17.1%,447.769063,9.558333,182.291667,4.000000
5,2026-07-29,24,€0.152,€0.000,€0.310,13.9%,494.550521,8.991667,300.041667,0.000000
6,2026-07-28,24,€0.140,€0.000,€0.250,18.1%,464.283021,8.879167,277.500000,0.000000
7,2026-07-26,24,€0.100,€0.000,€0.180,43.7%,322.474792,14.887500,67.875000,10.000000
8,2026-07-25,24,€0.109,€-0.010,€0.210,26.5%,425.362500,10.450000,288.833333,1.000000
9,2026-07-24,24,€0.159,€0.010,€0.240,8.5%,480.992708,4.579167,208.916667,3.000000


## Conclusion

The MySQL database now provides a reproducible SQL layer between the Python transformation and Tableau:

- `hourly_data` stores the validated historical observations;
- `vw_analysis_thresholds` defines cheap and low-carbon dynamically;
- `vw_hourly_dashboard` is the Tableau-ready row-level dataset;
- `vw_daily_summary_utc` and `vw_hourly_summary_utc` provide reusable aggregates;
- SQL quality checks confirm 1,170 unique validated hours and 164 cheap-and-low-carbon hours.

In [11]:
connection.close()
engine.dispose()
print("MySQL connection closed.")

MySQL connection closed.
